In [1]:
print("Hello, World!")

Hello, World!


In [2]:
import pandas as pd

file_path = "trainData.xlsx"   
all_sheets = pd.read_excel(file_path, sheet_name=None)

print("Sheet names:", list(all_sheets.keys()))


Sheet names: ['Gemini', 'ChatGPT', 'Tele', 'Telegram', 'DeepSeek']


In [3]:
# دمج كل الشيتات في DataFrame واحد
dfs = []

for sheet_name, sheet_df in all_sheets.items():
    if sheet_df is None or sheet_df.empty:
        continue

    sheet_df = sheet_df.copy()
    sheet_df["source"] = sheet_name  # تحديد مصدر الشيت
    dfs.append(sheet_df)

# دمج كل البيانات
df = pd.concat(dfs, ignore_index=True)

# عرض معلومات عامة
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

# عرض أول 8 صفوف
df.head(8)


Total rows: 2321
Total columns: 3


,نص الاستشارة,التصنيف,source
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,Gemini
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,Gemini
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,Gemini
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية,Gemini
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,Gemini
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية,Gemini
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية,Gemini
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,Gemini


In [4]:
import re

def clean_text(text):
    text = str(text)
    
    # حذف الروابط
    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # حذف الإيميلات
    text = re.sub(r"\S+@\S+", " ", text)
    
    # حذف الأرقام (عربي + إنجليزي)
    text = re.sub(r"[0-9٠-٩]", " ", text)
    
    # حذف الرموز الخاصة
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    
    # حذف المسافات الزائدة
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

# تطبيق التنظيف
df["clean_text"] = df["نص الاستشارة"].apply(clean_text)

df.head(8)


,نص الاستشارة,التصنيف,source,clean_text
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,Gemini,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,Gemini,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,Gemini,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية,Gemini,زوجي هجر البيت ولا يصرف على العيال من شهور
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,Gemini,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية,Gemini,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية,Gemini,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,Gemini,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...


In [5]:
def normalize_arabic(text):
    
    # إزالة التشكيل
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)
    
    # توحيد الهمزات
    text = re.sub(r"[إأآا]", "ا", text)
    
    # ى → ي
    text = re.sub(r"ى", "ي", text)
    
    # ة → ه (توحيد اختياري)
    text = re.sub(r"ة", "ه", text)
    
    # إزالة المد
    text = re.sub(r"ـ", "", text)
    
    # حذف مسافات إضافية
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

# تطبيق النورملزيشن
df["normalized_text"] = df["clean_text"].apply(normalize_arabic)

df.head(8)


,نص الاستشارة,التصنيف,source,clean_text,normalized_text
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,Gemini,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,شريكي سحب سيوله من المؤسسه بدون فواتير وش الحل؟
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,Gemini,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,صاحب العمل فصلني بدون سابق انذار ولا اعطاني مك...
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,Gemini,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب,تعرضت لابتزاز بصور خاصه من حساب وهمي في سناب
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية,Gemini,زوجي هجر البيت ولا يصرف على العيال من شهور,زوجي هجر البيت ولا يصرف علي العيال من شهور
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,Gemini,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,شريت شقه وطلعت فيها عيوب في السباكه والمالك ير...
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية,Gemini,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,تم استبعادي من مسابقه وظيفيه حكوميه رغم انطباق...
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية,Gemini,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه,البنك سحب مبلغ اكبر من القسط الشهري المتفق عليه
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,Gemini,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,اخوي الكبير رافض يوزع ورث ابوي ومستولي علي الم...


In [6]:
# حذف  للكلمات الشائعة
arabic_stopwords = set([
    "من", "في", "على", "الى", "إلى", "عن", "مع", "هذا", "هذه","السلام","عليكم","ورحمة","الله","رفع", "دعوى","عافيه"
    "الذي", "التي", "كان", "كانت", "جدا", "جداً", "لكن", "او", "أو"
])

def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word not in arabic_stopwords]
    return " ".join(filtered)

df["final_text"] = df["normalized_text"].apply(remove_stopwords)

df.head(8)


,نص الاستشارة,التصنيف,source,clean_text,normalized_text,final_text
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,Gemini,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,شريكي سحب سيوله من المؤسسه بدون فواتير وش الحل؟,شريكي سحب سيوله المؤسسه بدون فواتير وش الحل؟
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,Gemini,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,صاحب العمل فصلني بدون سابق انذار ولا اعطاني مك...,صاحب العمل فصلني بدون سابق انذار ولا اعطاني مك...
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,Gemini,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب,تعرضت لابتزاز بصور خاصه من حساب وهمي في سناب,تعرضت لابتزاز بصور خاصه حساب وهمي سناب
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية,Gemini,زوجي هجر البيت ولا يصرف على العيال من شهور,زوجي هجر البيت ولا يصرف علي العيال من شهور,زوجي هجر البيت ولا يصرف علي العيال شهور
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,Gemini,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,شريت شقه وطلعت فيها عيوب في السباكه والمالك ير...,شريت شقه وطلعت فيها عيوب السباكه والمالك يرفض ...
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية,Gemini,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,تم استبعادي من مسابقه وظيفيه حكوميه رغم انطباق...,تم استبعادي مسابقه وظيفيه حكوميه رغم انطباق ال...
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية,Gemini,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه,البنك سحب مبلغ اكبر من القسط الشهري المتفق عليه,البنك سحب مبلغ اكبر القسط الشهري المتفق عليه
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,Gemini,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,اخوي الكبير رافض يوزع ورث ابوي ومستولي علي الم...,اخوي الكبير رافض يوزع ورث ابوي ومستولي علي الم...


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

X = vectorizer.fit_transform(df["final_text"].fillna("").astype(str))

print("Shape of X:", X.shape)


Shape of X: (2321, 5000)


In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df["التصنيف"].astype(str))

print("Classes:", le.classes_)


Classes: ['القضايا الأسرية' 'القضايا الإدارية' 'القضايا التجارية'
 'القضايا الجنائية' 'القضايا العقارية' 'القضايا العمالية'
 'القضايا المالية' 'قضايا الأحوال الشخصية']


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])


Training size: 1856
Testing size: 465


In [10]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# تدريب المودل
model = LinearSVC(class_weight="balanced")
model.fit(X_train, y_train)

# التوقع على بيانات الاختبار
y_pred = model.predict(X_test)

# التقييم
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.8666666666666667

Classification Report:

                       precision    recall  f1-score   support

      القضايا الأسرية       0.71      0.61      0.65        28
     القضايا الإدارية       0.91      0.86      0.88        58
     القضايا التجارية       0.82      0.91      0.86        56
     القضايا الجنائية       0.96      0.83      0.89        54
     القضايا العقارية       0.90      0.94      0.92        66
     القضايا العمالية       0.97      0.92      0.94        92
      القضايا المالية       0.77      0.83      0.80        52
قضايا الأحوال الشخصية       0.78      0.85      0.81        59

             accuracy                           0.87       465
            macro avg       0.85      0.84      0.85       465
         weighted avg       0.87      0.87      0.87       465


Confusion Matrix:

[[17  0  1  0  0  0  0 10]
 [ 0 50  4  0  1  0  2  1]
 [ 0  0 51  0  2  1  2  0]
 [ 0  1  0 45  1  1  6  0]
 [ 0  0  2  0 62  1  0  1]
 [ 0  0  3  0  1 85  3  0]
 [ 2 

In [11]:
def predict_case(text):
    text = clean_text(text)
    text = normalize_arabic(text)
    text = remove_stopwords(text)
    
    vector = vectorizer.transform([text])
    prediction = model.predict(vector)
    
    return le.inverse_transform(prediction)[0]


In [12]:
predict_case("صاحب العمل لم يعطني راتبي منذ 3 أشهر")


'القضايا العمالية'

In [13]:
import numpy as np

# اسماء الفئات
classes = le.classes_

# نحصل رقم الفئتين
family_idx = np.where(classes == "القضايا الأسرية")[0][0]
personal_idx = np.where(classes == "قضايا الأحوال الشخصية")[0][0]

# نشوف وين صار الخلط بينهم
conf_matrix = confusion_matrix(y_test, y_pred)

print("أسرية تم تصنيفها كأحوال شخصية:", conf_matrix[family_idx][personal_idx])
print("أحوال شخصية تم تصنيفها كأسَرية:", conf_matrix[personal_idx][family_idx])


أسرية تم تصنيفها كأحوال شخصية: 10
أحوال شخصية تم تصنيفها كأسَرية: 5


In [14]:
# نسترجع النصوص من مجموعة الاختبار
test_indices = X_test.indices

wrong_cases = []

for i in range(len(y_test)):
    if (y_test[i] == family_idx and y_pred[i] == personal_idx) or \
       (y_test[i] == personal_idx and y_pred[i] == family_idx):
        wrong_cases.append(i)

print("عدد الحالات المتداخلة:", len(wrong_cases))


عدد الحالات المتداخلة: 15


In [15]:
# نرجع النصوص الأصلية من مجموعة الاختبار
import numpy as np

X_train_idx, X_test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

for i in wrong_cases[:5]:  # نعرض أول 5 فقط
    original_index = X_test_idx[i]
    print("النص:")
    print(df.iloc[original_index]["نص الاستشارة"])
    print("الحقيقة:", le.inverse_transform([y_test[i]])[0])
    print("توقع المودل:", le.inverse_transform([y_pred[i]])[0])
    print("-" * 50)


النص:
إثبات الرجعة بعد الطقة الأولى بدون علم الزوجة.
الحقيقة: قضايا الأحوال الشخصية
توقع المودل: القضايا الأسرية
--------------------------------------------------
النص:
كيف ارفع دعوى اثبات نسب للطفل؟
الحقيقة: القضايا الأسرية
توقع المودل: قضايا الأحوال الشخصية
--------------------------------------------------
النص:
رفع دعوى خلع والزوج يطالب باسترجاع المهر والهدايا.
الحقيقة: القضايا الأسرية
توقع المودل: قضايا الأحوال الشخصية
--------------------------------------------------
النص:
أمي أرملة وتريد الزواج، وإخوتي يرفضون ويعترضون، هل يمكنهم منعها قانونياً؟
الحقيقة: قضايا الأحوال الشخصية
توقع المودل: القضايا الأسرية
--------------------------------------------------
النص:
عند تقديم دعوى نفقة العدة طلبوا ان يكون الاطفال البالغين هم المدعي ايضا...
الحقيقة: قضايا الأحوال الشخصية
توقع المودل: القضايا الأسرية
--------------------------------------------------


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# --- تجهيز y من جديد (للتأكد) ---
le = LabelEncoder()
y = le.fit_transform(df["التصنيف"].astype(str))

# --- نقسم نصوص من جديد داخل نفس الخلية عشان كل شيء يكون متناسق ---
X_text = df["final_text"].fillna("").astype(str)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# --- Pipeline: TF-IDF + LinearSVC ---
pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(class_weight="balanced"))
])

# --- Grid للتجارب (كلمات + عبارات) ---
param_grid = [
    {
        "tfidf": [TfidfVectorizer(ngram_range=(1,2))],
        "tfidf__max_features": [5000, 8000, 12000],
        "tfidf__min_df": [1, 2, 3],
        "tfidf__sublinear_tf": [True, False],
        "clf__C": [0.5, 1, 2, 4],
    },
    {
        # Char n-grams (قوي جدًا للعربي)
        "tfidf": [TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5))],
        "tfidf__max_features": [8000, 12000, 20000],
        "tfidf__min_df": [1, 2, 3],
        "tfidf__sublinear_tf": [True, False],
        "clf__C": [0.5, 1, 2, 4],
    }
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train_text, y_train)

print("Best params:", grid.best_params_)
print("Best CV f1_macro:", grid.best_score_)

best_model = grid.best_estimator_

# تقييم على test
y_pred = best_model.predict(X_test_text)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nTest Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))


Fitting 5 folds for each of 144 candidates, totalling 720 fits
Best params: {'clf__C': 0.5, 'tfidf': TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5)), 'tfidf__max_features': 20000, 'tfidf__min_df': 1, 'tfidf__sublinear_tf': True}
Best CV f1_macro: 0.839652979233302

Test Accuracy: 0.8731182795698925

Test Report:

                       precision    recall  f1-score   support

      القضايا الأسرية       0.74      0.61      0.67        28
     القضايا الإدارية       0.88      0.88      0.88        58
     القضايا التجارية       0.84      0.88      0.86        56
     القضايا الجنائية       0.98      0.85      0.91        54
     القضايا العقارية       0.87      0.94      0.91        66
     القضايا العمالية       0.97      0.96      0.96        92
      القضايا المالية       0.80      0.83      0.81        52
قضايا الأحوال الشخصية       0.79      0.85      0.82        59

             accuracy                           0.87       465
            macro avg       0.86      0.85   

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

FAMILY = "القضايا الأسرية"
PERSONAL = "قضايا الأحوال الشخصية"

# فلترة بيانات الفئتين فقط
df_fp = df[df["التصنيف"].astype(str).isin([FAMILY, PERSONAL])].copy()

X_fp = df_fp["final_text"].fillna("").astype(str)
y_fp = df_fp["التصنيف"].astype(str).values  # نخليه نصوص (فئتين فقط)

X_train_fp, X_test_fp, y_train_fp, y_test_fp = train_test_split(
    X_fp, y_fp, test_size=0.2, random_state=42, stratify=y_fp
)

# Pipeline للمودل المتخصص
pipe_fp = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(class_weight="balanced"))
])

# Grid أصغر (سريع) لكنه قوي
param_grid_fp = {
    "tfidf__analyzer": ["char_wb", "word"],
    "tfidf__ngram_range": [(3,5), (4,6), (1,2)],
    "tfidf__max_features": [5000, 10000, 20000],
    "tfidf__min_df": [1, 2],
    "tfidf__sublinear_tf": [True],
    "clf__C": [0.25, 0.5, 1, 2]
}

cv_fp = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_fp = GridSearchCV(
    pipe_fp,
    param_grid=param_grid_fp,
    scoring="f1_macro",
    cv=cv_fp,
    n_jobs=-1,
    verbose=2
)

grid_fp.fit(X_train_fp, y_train_fp)

specialist_model = grid_fp.best_estimator_

print("Specialist Best params:", grid_fp.best_params_)
print("Specialist Best CV f1_macro:", grid_fp.best_score_)

# تقييم المتخصص
y_pred_fp = specialist_model.predict(X_test_fp)
print("\nSpecialist Test Accuracy:", accuracy_score(y_test_fp, y_pred_fp))
print("\nSpecialist Report:\n")
print(classification_report(y_test_fp, y_pred_fp))


Fitting 5 folds for each of 144 candidates, totalling 720 fits
Specialist Best params: {'clf__C': 0.5, 'tfidf__analyzer': 'word', 'tfidf__max_features': 5000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Specialist Best CV f1_macro: 0.8023694653299916

Specialist Test Accuracy: 0.8181818181818182

Specialist Report:

                       precision    recall  f1-score   support

      القضايا الأسرية       0.73      0.68      0.70        28
قضايا الأحوال الشخصية       0.85      0.88      0.87        60

             accuracy                           0.82        88
            macro avg       0.79      0.78      0.79        88
         weighted avg       0.82      0.82      0.82        88



In [18]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# نفس أفضل إعداداتك من GridSearch العام:
general_model = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5),
                              max_features=20000, min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC(C=0.5, class_weight="balanced"))
])

X_text = df["final_text"].fillna("").astype(str)
y_text = df["التصنيف"].astype(str)

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
)

general_model.fit(X_train_text, y_train_text)

y_pred_general = general_model.predict(X_test_text)

print("General(Text) Accuracy:", accuracy_score(y_test_text, y_pred_general))
print(classification_report(y_test_text, y_pred_general))


General(Text) Accuracy: 0.8774193548387097
                       precision    recall  f1-score   support

      القضايا الأسرية       0.78      0.64      0.71        28
     القضايا الإدارية       0.88      0.88      0.88        58
     القضايا التجارية       0.83      0.88      0.85        56
     القضايا الجنائية       0.98      0.85      0.91        54
     القضايا العقارية       0.87      0.94      0.91        66
     القضايا العمالية       0.97      0.97      0.97        92
      القضايا المالية       0.83      0.83      0.83        52
قضايا الأحوال الشخصية       0.79      0.85      0.82        59

             accuracy                           0.88       465
            macro avg       0.87      0.85      0.86       465
         weighted avg       0.88      0.88      0.88       465



In [19]:
FAMILY = "القضايا الأسرية"
PERSONAL = "قضايا الأحوال الشخصية"

def preprocess_for_model(text: str) -> str:
    t = clean_text(text)
    t = normalize_arabic(t)
    t = remove_stopwords(t)
    return t

def predict_hierarchical(text: str) -> str:
    t = preprocess_for_model(text)

    pred1 = general_model.predict([t])[0]   # <-- نص عربي

    if pred1 in [FAMILY, PERSONAL]:
        pred2 = specialist_model.predict([t])[0]  # <-- نص عربي
        return pred2

    return pred1


In [20]:
from sklearn.metrics import confusion_matrix

y_pred_h = [predict_hierarchical(x) for x in X_test_text]

print("Hierarchical Accuracy:", accuracy_score(y_test_text, y_pred_h))
print(classification_report(y_test_text, y_pred_h))


Hierarchical Accuracy: 0.886021505376344
                       precision    recall  f1-score   support

      القضايا الأسرية       0.86      0.68      0.76        28
     القضايا الإدارية       0.88      0.88      0.88        58
     القضايا التجارية       0.83      0.88      0.85        56
     القضايا الجنائية       0.98      0.85      0.91        54
     القضايا العقارية       0.87      0.94      0.91        66
     القضايا العمالية       0.97      0.97      0.97        92
      القضايا المالية       0.83      0.83      0.83        52
قضايا الأحوال الشخصية       0.83      0.90      0.86        59

             accuracy                           0.89       465
            macro avg       0.88      0.86      0.87       465
         weighted avg       0.89      0.89      0.89       465



In [21]:
labels = sorted(y_test_text.unique())
cm = confusion_matrix(y_test_text, y_pred_h, labels=labels)

fam_i = labels.index(FAMILY)
per_i = labels.index(PERSONAL)

print("أسرية → أحوال شخصية:", cm[fam_i][per_i])
print("أحوال شخصية → أسرية:", cm[per_i][fam_i])


أسرية → أحوال شخصية: 8
أحوال شخصية → أسرية: 1


In [22]:
import joblib

joblib.dump(general_model, "general_model.joblib")
joblib.dump(specialist_model, "family_personal_specialist.joblib")

print("Saved models successfully.")


Saved models successfully.


In [23]:
FAMILY = "القضايا الأسرية"
PERSONAL = "قضايا الأحوال الشخصية"

def predict_case_final(text: str) -> str:
    t = preprocess_for_model(text)

    pred1 = general_model.predict([t])[0]

    if pred1 in [FAMILY, PERSONAL]:
        return specialist_model.predict([t])[0]

    return pred1


In [24]:
tests = [
    "أخي يرفض توزيع ميراث والدي، كيف أطلع صك ورثة؟",
    "زوجي ما يصرف علي من 4 شهور وأبغى نفقة",
    "تعرضت لابتزاز بصور في سناب",
    "صاحب العمل فصلني بدون إنذار",
]

for t in tests:
    print(t)
    print("=>", predict_case_final(t))
    print("-"*60)


أخي يرفض توزيع ميراث والدي، كيف أطلع صك ورثة؟
=> القضايا العقارية
------------------------------------------------------------
زوجي ما يصرف علي من 4 شهور وأبغى نفقة
=> القضايا الأسرية
------------------------------------------------------------
تعرضت لابتزاز بصور في سناب
=> القضايا الجنائية
------------------------------------------------------------
صاحب العمل فصلني بدون إنذار
=> القضايا العمالية
------------------------------------------------------------
